# Structured Outputs

**Prerequisites:** `01_your_first_llm.ipynb`, `04_prompting.ipynb`

Almost every agent needs to turn an LLM reply into a Python object it can act on — not just print. This notebook builds that pipeline: JSON mode, a Pydantic schema, validation, and a retry loop for when validation fails, because on this model it will.

**By the end you'll have:**
- Seen free-text output vary in structure across identical calls
- Used JSON mode, and confirmed exactly what it does and doesn't guarantee
- Triggered a real JSON parsing failure
- Validated a reply against a Pydantic schema, and seen it correctly reject bad data
- Built `extract()` in `llm.py` — JSON mode + validation + retry, wrapped up

In [ ]:
import sys
import json
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from pydantic import BaseModel, ValidationError

from llm import chat, extract, DEFAULT_MODEL

## Why this matters

If you ask for something in plain English and just eyeball the reply, it looks fine. The problem shows up when *code*, not you, has to parse it — and the exact same prompt doesn't reliably produce the exact same structure twice:

In [ ]:
for _ in range(3):
    reply = chat(
        [{"role": "user", "content": "Give me a recipe for pancakes with a title, ingredients, and total time."}],
        temperature=1,
    )
    print(repr(reply[:80]))

Headers, formatting, even whether "Ingredients" gets a `###` in front of it — all of that drifts run to run. Any regex or string-splitting logic you write against one of these will break on another. That's the actual problem structured output solves; it's not about making things "look nicer."

## JSON mode

`response_format={"type": "json_object"}` constrains the model to emit syntactically valid JSON — nothing more.

In [ ]:
reply = chat(
    [{"role": "user", "content": (
        "Give me a recipe for pancakes as JSON with fields "
        "title, ingredients (list), minutes (int)."
    )}],
    response_format={"type": "json_object"},
)
print(reply)

Two things worth knowing, both from hitting them directly:

1. **The word "json" has to appear somewhere in your messages**, or Groq (like OpenAI) rejects the request outright with a 400 — `response_format` alone isn't enough.
2. **"Valid JSON" isn't "matches my schema."** JSON mode guarantees the syntax parses; it says nothing about which fields exist or what type they are. `llama-3.3-70b-versatile` doesn't support Groq's stricter `json_schema` mode either — as of writing, that's limited to `openai/gpt-oss-20b` and `openai/gpt-oss-120b` on Groq. So on this model, syntax is guaranteed, structure isn't — which is exactly what the rest of this notebook deals with.

## Parsing failures

Drop `response_format` and just ask nicely — a real, common failure mode shows up:

In [ ]:
unstructured_reply = chat([{"role": "user", "content": (
    "Give me a recipe for pancakes as JSON with fields title, ingredients (list), minutes (int)."
)}])
print(repr(unstructured_reply[:60]))

try:
    json.loads(unstructured_reply)
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e)

The model wrapped the JSON in a ` ```json ` markdown fence — helpful in a chat UI, fatal to `json.loads()`. This is exactly why JSON mode exists: without it, even asking nicely for JSON doesn't reliably produce parseable output. JSON mode fixes this specific failure — but as noted above, it doesn't guarantee the parsed data matches your schema, which is what validation, next, is for.

## Pydantic models

A schema says precisely what "correct" means, and gives you a typed object back instead of a dict you're hoping has the right keys:

In [ ]:
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    minutes: int

In [ ]:
recipe = Recipe.model_validate_json(reply)
print(type(recipe))
print(recipe)
print(recipe.minutes, "minutes,", len(recipe.ingredients), "ingredients")

That's the actual point: `recipe.minutes` is a real `int` you can do arithmetic on, `recipe.ingredients` is a real `list`, not text you'd have to re-parse.

## Validation catches real problems

JSON mode would happily accept this — it's syntactically valid JSON. It just doesn't match `Recipe`:

In [ ]:
bad_json = '{"title": "Pancakes", "ingredients": "flour, milk", "minutes": "twenty"}'

try:
    Recipe.model_validate_json(bad_json)
except ValidationError as e:
    print(e)

Two independent errors, each naming the exact field and what was wrong with it: `ingredients` was a string instead of a list, `minutes` was a string that isn't even a valid integer. This is what "validation" buys you over just calling `json.loads()` — a schema mismatch is caught immediately, with a precise, actionable error, instead of surfacing later as a confusing bug somewhere downstream.

## Retries

The idea: call, try to validate, and if it fails, tell the model *exactly* what was wrong and ask it to fix it, using the real error message as feedback. Walked through by hand — suppose attempt 1 comes back like this:

In [ ]:
attempt_1 = '{"title": "Pancakes", "ingredients": "flour, milk, sugar", "minutes": 20}'

try:
    Recipe.model_validate_json(attempt_1)
except ValidationError as e:
    error = str(e)
    print("attempt 1 failed:")
    print(error)

In [ ]:
messages = [
    {"role": "user", "content": "Give me a recipe for pancakes as JSON."},
    {"role": "assistant", "content": attempt_1},
    {"role": "user", "content": f"That was invalid: {error}\nReturn corrected JSON only, matching the schema."},
]

attempt_2 = chat(messages, response_format={"type": "json_object"})
recipe = Recipe.model_validate_json(attempt_2)
print("attempt 2 succeeded:", recipe)

Feeding the real validation error back as a follow-up message is enough for the model to fix its own mistake. Looping that — call, validate, feed the error back on failure, stop after N attempts — is `extract()` in `llm.py`:

```python
def extract(prompt, schema, retries=3, model=DEFAULT_MODEL):
    messages = [{"role": "user", "content": prompt}]
    for _ in range(retries):
        reply = chat(messages, model=model, response_format={"type": "json_object"})
        messages.append({"role": "assistant", "content": reply})
        try:
            return schema.model_validate_json(reply)
        except (json.JSONDecodeError, ValidationError) as e:
            messages.append({
                "role": "user",
                "content": f"That was invalid: {e}\nReturn corrected JSON only, matching the schema.",
            })
    raise ValueError(f"Failed to get valid {schema.__name__} after {retries} attempts")
```

In [ ]:
recipe = extract("Give me a recipe for pancakes, as JSON.", Recipe)
print(type(recipe))
print(recipe)

## What you built

✓ Watched free-text formatting actually vary across identical calls

✓ Used JSON mode and confirmed precisely what it does and doesn't guarantee

✓ Triggered a real JSON parsing failure from markdown-fenced output

✓ Defined `Recipe` and validated against it, and seen validation correctly reject bad data with field-level errors

✓ Built `extract()` in `llm.py` — JSON mode, validation, and retry-with-feedback in one call

**Next:** `06_function_calling.ipynb` — tool schemas and a single call→execute→result round trip.